# 🥾 Side Trails · Step 4 — More Kernels
**A Billion Small Steps · Hardware for AI · Step 4 · Book sections App. D.2 (from 4.3) · Foothold lessons 4.3**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

| Part | What you do | Book |
|---|---|---|
| A | A 2-D kernel: brighten an image, one thread per pixel | App. D.2 |

⏱️ About 30 minutes. 📘 Textbook: Hwu, Kirk & El Hajj Ch 3. Course site: https://aishugithub.github.io/course_GPU/

🥾 **Side Trail: leaves from 4.3 → rejoin at 4.4.**

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

---
## Part A · A 2-D kernel: brighten an image  *(Appendix D.2)*
One thread per pixel, with 16 × 16 blocks: each thread finds its row and column from `blockIdx`, `blockDim` and `threadIdx` in two dimensions.

In [ ]:
bright_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void brighten(float *img, int width, int height, float amount) {   // add "amount" to every pixel of a width × height image
    // this thread's column (x direction) and row (y direction)
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;   // and this thread's row
    // the safety check in both directions
    if (col < width && row < height)
        // pixel (row, col) is at position row × width + col; fminf caps it at 1.0 (white)
        img[row * width + col] = fminf(img[row * width + col] + amount, 1.0f);
}
'''
# a made-up 480 × 640 grey picture: a wavy pattern of brightness 0..0.8
H, W = 480, 640
yy, xx = np.mgrid[0:H, 0:W]   # the row and column number of every pixel
image = (0.4 * np.sin(xx / 40) * np.cos(yy / 30) + 0.4).astype(np.float32)   # a wavy pattern of grey levels

bright_k = cp.RawKernel(bright_src, "brighten")   # compile the kernel
# copy the image to the GPU; blocks of 16 × 16 = 256 threads; enough blocks across and down (rounded up)
d_img = cp.asarray(image)
block = (16, 16)   # blocks of 16 × 16 = 256 threads
grid = ((W + 15) // 16, (H + 15) // 16)   # blocks across and down: (40, 30)
# launch: add 0.35 to every pixel, then copy the result back
bright_k(grid, block, (d_img, cp.int32(W), cp.int32(H), cp.float32(0.35)))
result = cp.asnumpy(d_img)   # copy the result back to the CPU
print(f"grid = {grid} blocks of {block} threads = {grid[0] * grid[1] * 256:,} threads for {H * W:,} pixels")   # one thread per pixel

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))      # two pictures side by side
# vmin/vmax fix the grey scale to 0..1, so "before" and "after" use the same brightness scale
ax[0].imshow(image, cmap="gray", vmin=0, vmax=1); ax[0].set_title("before"); ax[0].axis("off")                        # the original
ax[1].imshow(result, cmap="gray", vmin=0, vmax=1); ax[1].set_title("after: brightened on the GPU"); ax[1].axis("off")  # the result
plt.show()                                          # draw them

### ✏️ YOUR TURN (A)
1. Change the kernel to make the image **darker**. Which single line did you change?
2. The image is not a multiple of 16 pixels wide. Which line stops the threads past the edge from writing?

*Your answers:*